# Phase 6 — Packaged search service
Viewer over `service_manifest.json` and `artifacts/reports/search_service_report.json`. No search is run here; use
`python scripts/search_cli.py "query"` or `SearchService.load(...)`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "base.yaml").is_file())
sys.path.insert(0, str(root))
from src.config import load_config
from src.utils import read_json
cfg = load_config(["configs/baseline.yaml", "configs/compare.yaml", "configs/service.yaml"], root=root)
pkg = cfg["_abs"]["service_dir"] / cfg["service"]["package_name"]
mp, rp = pkg / "service_manifest.json", cfg["_abs"]["reports_dir"] / "search_service_report.json"
MAN = read_json(mp) if mp.is_file() else None
REP = read_json(rp) if rp.is_file() else None
print("package :", "built" if MAN else "NOT BUILT (python scripts/build_search_package.py)")
print("validation:", REP["status"] if REP else "NOT RUN (python scripts/validate_search_service.py)")

package : built
validation: PASS


## 1. Package contents and provenance

In [2]:
if MAN:
    display(pd.Series({k: MAN[k] for k in ("package_name", "selected_model", "label", "n_documents", "embedding_dim", "max_seq_length", "normalization", "metric", "index_type", "created_utc")}, name="value").to_frame())
    print("freeze fingerprint:", MAN["freeze_fingerprint"]); print("weights sha256   :", MAN["weights_sha256"])
    print("reported metrics (from Phase 5, not recomputed here):")
    display(pd.DataFrame({k: v for k, v in MAN["reported_metrics"].items() if k != "source"}).loc[["ndcg@10", "recall@1", "recall@10", "mrr@10"]].round(4))
    print(MAN["notice"])

,value
package_name,scifact_search_v1
selected_model,ft_hardneg_dense
label,Fine-tuned (+ mined dense hard negatives)
n_documents,5183
embedding_dim,384
max_seq_length,256
normalization,L2 (queries and documents)
metric,cosine == inner product
index_type,IndexFlatIP (exact)
created_utc,2026-10-26T06:43:02+00:00


freeze fingerprint: d8e886cb5d080d5f34aba4111ca73d4546160191666e4cfb5fa9893e949ad0db
weights sha256   : 405b247000543b91781bc114afdecbfe7e1e3133f6dae62d352717a949efc051
reported metrics (from Phase 5, not recomputed here):


,heldout_test,validation
ndcg@10,0.7131,0.7433
recall@1,0.5782,0.6183
recall@10,0.8277,0.8323
mrr@10,0.6815,0.7174


similarity_score is cosine similarity, not a probability of relevance or factual correctness.


## 2. Exit gates (all must be True)

In [3]:
if REP: display(pd.Series(REP["gates"], name="passed").to_frame())

,passed
faiss_index_equals_packaged_embeddings,True
input:default_top_k_used_when_none,True
input:empty_and_whitespace_query_return_no_hits_gracefully,True
input:huge_top_k_is_clipped_and_flagged,True
input:invalid_top_k_raises,True
input:non_string_query_raises_TypeError,True
input:nul_character_handled,True
input:unicode_query_ok,True
input:very_long_query_truncated_not_crashing,True
input:whitespace_variants_give_identical_results,True


## 3. Reload consistency vs Phase 5 rankings (tie-aware)

In [4]:
if REP:
    print(REP["reload_consistency"]); print(REP["end_to_end_metric"]); print("max |service score - independent dot product|:", REP["score_check_max_abs_diff"])

{'compared_depth': 100, 'exact': 162, 'failing_queries': [], 'n_queries': 162, 'not_equivalent': 0, 'tie_aware_equivalent_not_exact': 0, 'tie_tolerance': 2e-05}
{'abs_diff': 0.0, 'note': 'validation split only; test.tsv not used in this phase', 'phase5_val_ndcg@10': 0.743263648033353, 'service_val_ndcg@10': 0.743263648033353}
max |service score - independent dot product|: 1.1920928955078125e-07


## 4. Latency AFTER startup (CPU, sequential, one user) and startup cost

In [5]:
if REP:
    rows = {k: {**{f"total {s} (ms)": v for s, v in b["total_ms"].items()}, "encode mean (ms)": b["encode_ms_mean"], "FAISS mean (ms)": b["faiss_search_ms_mean"], "format mean (ms)": b["format_ms_mean"], "queries/s": b["throughput_qps_sequential"]} for k, b in REP["latency_after_startup"].items()}
    display(pd.DataFrame(rows).T.round(2))
    print("startup seconds (EXCLUDED from the latency above):", {k: round(v, 2) for k, v in REP["startup_s_excluded_from_latency"].items()})
    print("memory:", REP["memory_mb"]); print("package:", REP["package"]); print("environment:", REP["environment"])

,total max (ms),total mean (ms),total p50 (ms),total p95 (ms),total p99 (ms),encode mean (ms),FAISS mean (ms),format mean (ms),queries/s
top_k=1,87.79,16.51,15.53,22.87,29.74,15.50,0.86,0.15,60.43
top_k=10,102.39,18.32,17.26,23.83,32.62,16.43,0.88,1.01,54.45
top_k=100,105.82,23.90,23.34,30.49,42.14,14.41,0.80,8.68,41.73


startup seconds (EXCLUDED from the latency above): {'load_documents': 0.06, 'load_index': 0.04, 'load_model': 17.13, 'total': 17.81, 'verify_hashes': 0.59}
memory: {'note': 'RSS of the whole Python process (two services were loaded for the reload check)', 'peak_rss_during_benchmark': 674.9, 'rss_after_load': 500.5}
package: {'extra_files': [], 'n_files': 13, 'size_mb': 115.6}
environment: {'cpu_count': 8, 'torch_threads': 4}


## 5. Observed input handling

In [6]:
if REP: display(pd.Series(REP["input_handling_observed"], name="behaviour").to_frame())

,behaviour
empty / whitespace query,"status=empty_query, 0 hits (no exception)"
top_k 0 / negative,ValueError
"top_k = 1,000,000",clipped to 100 with warning
top_k float / bool / str,TypeError


## 6. Example queries (retrieval demo — similarity is not a probability that a claim is true)

In [7]:
if REP:
    for q, hits in REP["example_queries"].items():
        print(q)
        for h in hits: print(f"  #{h['rank']} [{h['corpus_id']}] cos={h['similarity_score']:.3f}  {h['title'][:90]}")

CRISPR off-target effects in human cells
  #1 [47018050] cos=0.560  CRISPR–Cas9 genome editing induces a p53-mediated DNA damage response
  #2 [461550] cos=0.546  Multiplex genome engineering using CRISPR/Cas systems.
  #3 [21561474] cos=0.483  Increasing the efficiency of precise genome editing with CRISPR-Cas9 by inhibition of nonh
Wnt signaling in embryonic stem cell differentiation
  #1 [23418635] cos=0.632  Embryonic stem cells require Wnt proteins to prevent differentiation to epiblast stem cell
  #2 [23804187] cos=0.614  Wnt/β-catenin signaling defines organizing centers that orchestrate growth and differentia
  #3 [30675656] cos=0.607  Dishevelled Activates JNK and Discriminates between JNK Pathways in Planar Polarity and wi
statin therapy and cardiovascular mortality
  #1 [5698494] cos=0.653  The benefits of statins in people without established cardiovascular disease but with card
  #2 [1287809] cos=0.617  Cost-effectiveness of 10-Year Risk Thresholds for Initiation of Statin

## 7. Caveats
- Latency is for one CPU machine, sequential single-user calls; it is not a throughput or multi-user guarantee.
- The service searches only the 5,183 SciFact abstracts it was built on; it has no knowledge beyond them.
- Results reflect semantic similarity to known abstracts, not verification of any claim.